In [1]:
import numpy as np
import pandas as pd

from xgboost import XGBRegressor

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
from google.colab import files

uploaded = files.upload()

Saving test_dataset.csv to test_dataset.csv
Saving train_dataset.csv to train_dataset.csv


In [3]:
train = pd.read_csv("train_dataset.csv")
test = pd.read_csv("test_dataset.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (150, 6)
Test shape: (50, 5)


In [4]:
print("TRAIN")
print(train.info())

print("\nMissing values:")
print(train.isnull().sum())

print("\nDuplicate rows:", train.duplicated().sum())

TRAIN
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   flow_rate_L_min       150 non-null    float64
 1   concentration_mol_L   150 non-null    float64
 2   inlet_temperature_K   150 non-null    float64
 3   length_m              150 non-null    float64
 4   jacket_temperature_K  150 non-null    float64
 5   overall_yield         150 non-null    float64
dtypes: float64(6)
memory usage: 7.2 KB
None

Missing values:
flow_rate_L_min         0
concentration_mol_L     0
inlet_temperature_K     0
length_m                0
jacket_temperature_K    0
overall_yield           0
dtype: int64

Duplicate rows: 0


In [5]:
X = train.drop(columns=["overall_yield"])
y = train["overall_yield"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (150, 5)
Target shape: (150,)


In [6]:
X_final = X.copy()

X_final["residence_proxy"] = (
    X_final["length_m"] /
    X_final["flow_rate_L_min"]
)

X_final["mean_T"] = (
    X_final["inlet_temperature_K"] +
    X_final["jacket_temperature_K"]
) / 2

print("Final feature shape:", X_final.shape)
display(X_final.head())

Final feature shape: (150, 7)


,flow_rate_L_min,concentration_mol_L,inlet_temperature_K,length_m,jacket_temperature_K,residence_proxy,mean_T
0,33.09,3.68,357.75,19.87,383.79,0.600484,370.770
1,76.30,1.34,429.70,14.84,405.72,0.194495,417.710
2,59.90,1.01,431.10,11.76,385.40,0.196327,408.250
3,49.90,2.21,445.61,22.85,367.74,0.457916,406.675
4,16.70,3.95,458.91,4.56,374.13,0.273054,416.520


In [7]:
X_test_final = test.copy()

X_test_final["residence_proxy"] = (
    X_test_final["length_m"] /
    X_test_final["flow_rate_L_min"]
)

X_test_final["mean_T"] = (
    X_test_final["inlet_temperature_K"] +
    X_test_final["jacket_temperature_K"]
) / 2

print("Test feature shape:", X_test_final.shape)
display(X_test_final.head())

Test feature shape: (50, 7)


,flow_rate_L_min,concentration_mol_L,inlet_temperature_K,length_m,jacket_temperature_K,residence_proxy,mean_T
0,43.73,2.88,454.18,7.66,440.61,0.175166,447.395
1,47.80,2.33,401.37,19.95,390.13,0.417364,395.750
2,7.14,0.65,411.86,3.05,476.23,0.427171,444.045
3,17.86,1.28,385.97,22.36,353.56,1.251960,369.765
4,56.40,2.51,495.33,8.36,517.71,0.148227,506.520


In [8]:
print("Train columns:")
print(X_final.columns.tolist())

print("\nTest columns:")
print(X_test_final.columns.tolist())

print("\nColumns match:",
      X_final.columns.tolist() == X_test_final.columns.tolist())

Train columns:
['flow_rate_L_min', 'concentration_mol_L', 'inlet_temperature_K', 'length_m', 'jacket_temperature_K', 'residence_proxy', 'mean_T']

Test columns:
['flow_rate_L_min', 'concentration_mol_L', 'inlet_temperature_K', 'length_m', 'jacket_temperature_K', 'residence_proxy', 'mean_T']

Columns match: True


## Model Selection

The XGBoost model was developed using repeated 5-fold cross-validation.

Feature engineering experiments showed that:
- `residence_proxy` improved the baseline model.
- `mean_T` provided a further improvement.
- `delta_T` did not improve repeated CV performance and was therefore removed.
- `tau_meanT` also reduced performance and was removed.

Selected features:
- Original 5 process variables
- residence_proxy
- mean_T

Selected XGBoost hyperparameters:
- n_estimators = 500
- learning_rate = 0.05
- max_depth = 3
- min_child_weight = 5

Repeated 5-fold CV RMSE:
18.79

After validating physical target clipping to [0, 100]:
18.64 RMSE


In [9]:
final_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=3,
    min_child_weight=5,
    random_state=42
)

final_model.fit(X_final, y)

print("Final XGBoost model trained successfully!")

Final XGBoost model trained successfully!


In [10]:
test_predictions = final_model.predict(X_test_final)

print("Number of predictions:", len(test_predictions))
print("Minimum prediction:", test_predictions.min())
print("Maximum prediction:", test_predictions.max())
print("Mean prediction:", test_predictions.mean())

Number of predictions: 50
Minimum prediction: -6.395308
Maximum prediction: 90.37496
Mean prediction: 28.328773


In [11]:
final_predictions = np.clip(
    test_predictions,
    0,
    100
)

print("After clipping:")
print("Minimum:", final_predictions.min())
print("Maximum:", final_predictions.max())
print("Mean:", final_predictions.mean())

After clipping:
Minimum: 0.0
Maximum: 90.37496
Mean: 28.850544


In [12]:
submission = pd.DataFrame({
    "overall_yield": final_predictions
})

submission.to_csv(
    "XGBoost_submission.csv",
    index=False
)

print("Submission created!")
print("Shape:", submission.shape)

display(submission.head(10))

Submission created!
Shape: (50, 1)


,overall_yield
0,21.459084
1,90.374962
2,1.945166
3,68.727074
4,13.373159
5,73.166809
6,84.535461
7,84.956551
8,2.715322
9,76.478020


In [13]:
print("Rows:", len(submission))
print("Columns:", submission.columns.tolist())
print("Missing values:")
print(submission.isnull().sum())

print("\nPrediction range:")
print("Min:", submission["overall_yield"].min())
print("Max:", submission["overall_yield"].max())

Rows: 50
Columns: ['overall_yield']
Missing values:
overall_yield    0
dtype: int64

Prediction range:
Min: 0.0
Max: 90.37496185302734


In [14]:
from google.colab import files

files.download("XGBoost_submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>